# Exercise 5.1: Capstone — Complete Flight Disruption Agent

**Module:** 5 — Capstone
**Level:** Advanced

You've learned chains, memory, graphs, and conditional routing. Now you'll build a **complete agent** that combines everything into a production-like flight disruption management system.

**This notebook has less hand-holding.** You know the building blocks — now we focus on architecture.

**Architecture overview:**
```
START → intake → classify → route_by_severity
  ├── high   → emergency_response → draft_comms → quality_check → END
  ├── medium → standard_response  → draft_comms → quality_check → END
  └── low    → auto_response      → draft_comms → quality_check → END
```

## 1. Setup

Get your free Groq API key at: https://console.groq.com/keys

In [ ]:
# Install all required packages
!pip install langgraph langchain langchain-groq -q

In [ ]:
import os

# Paste your Groq API key between the quotes
# Get it free at: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = "your-groq-key-here"

## 2. Agent State

The state is the agent's "working memory" — it accumulates data as the agent processes a disruption. We use `Annotated` with `operator.add` for the `actions_taken` field so that each node's actions are **appended** rather than overwritten.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
import operator


class AgentState(TypedDict):
    # Input
    raw_report: str                                     # The incoming disruption report

    # Intake processing
    flight_number: str                                  # Extracted flight number
    disruption_type: str                                # delay, cancellation, diversion
    structured_info: str                                # Cleaned and structured report

    # Classification
    severity: str                                       # low, medium, high
    priority_score: int                                 # 1-10 priority score

    # Resolution
    resolution_plan: str                                # The chosen resolution
    actions_taken: Annotated[list[str], operator.add]   # Log of all actions (appended by each node)

    # Communication
    passenger_notification: str                         # Final passenger message
    internal_report: str                                # Report for operations team

    # Quality gate
    quality_score: int                                  # 1-10 quality rating
    quality_feedback: str                               # Feedback on the communications


# Create the model
model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("Agent state defined with 12 fields.")
print("The actions_taken list APPENDS — each node adds to it, nothing is lost.")

## 3. Node Definitions

Each node handles one responsibility. The agent is built from simple, focused functions.

In [ ]:
def intake_node(state: AgentState) -> dict:
    """Parse and structure the raw disruption report."""
    raw = state["raw_report"]

    response = model.invoke(
        f"""Parse this disruption report and extract key information.
Respond in this EXACT format (one field per line):

FLIGHT: [flight number]
TYPE: [delay/cancellation/diversion]
STRUCTURED: [clean summary with all relevant details in 3-4 sentences]

Report:
{raw}"""
    )

    content = response.content

    # Parse the structured response
    flight_number = "UNKNOWN"
    disruption_type = "unknown"
    structured = content

    for line in content.split("\n"):
        line_upper = line.strip().upper()
        if line_upper.startswith("FLIGHT:"):
            flight_number = line.split(":", 1)[1].strip()
        elif line_upper.startswith("TYPE:"):
            disruption_type = line.split(":", 1)[1].strip().lower()
        elif line_upper.startswith("STRUCTURED:"):
            structured = line.split(":", 1)[1].strip()

    return {
        "flight_number": flight_number,
        "disruption_type": disruption_type,
        "structured_info": structured,
        "actions_taken": [f"Intake: parsed report for {flight_number} ({disruption_type})"]
    }


print("intake_node: parses raw report into structured fields")

In [ ]:
def classify_node(state: AgentState) -> dict:
    """Classify severity and assign priority score."""
    info = state["structured_info"]
    d_type = state["disruption_type"]

    response = model.invoke(
        f"""Classify this {d_type} disruption. Respond in EXACT format:

SEVERITY: [low/medium/high]
PRIORITY: [1-10 integer]

Scoring guide:
- Cancellations: high severity, priority 8-10
- Delays > 3 hours or with many connections: high severity, priority 7-9
- Delays 1-3 hours: medium severity, priority 4-6
- Delays < 1 hour, minor issues: low severity, priority 1-3

Disruption: {info}"""
    )

    content = response.content
    severity = "medium"
    priority = 5

    for line in content.split("\n"):
        line_upper = line.strip().upper()
        if line_upper.startswith("SEVERITY:"):
            parsed = line.split(":", 1)[1].strip().lower()
            if parsed in ["low", "medium", "high"]:
                severity = parsed
        elif line_upper.startswith("PRIORITY:"):
            try:
                priority = int(line.split(":", 1)[1].strip())
            except ValueError:
                priority = 5  # Default if parsing fails

    return {
        "severity": severity,
        "priority_score": priority,
        "actions_taken": [f"Classify: severity={severity}, priority={priority}"]
    }


print("classify_node: determines severity and priority score")

In [ ]:
def emergency_response_node(state: AgentState) -> dict:
    """Handle high-severity disruptions with full emergency protocol."""
    info = state["structured_info"]

    response = model.invoke(
        f"""EMERGENCY PROTOCOL for high-severity disruption.
Create a detailed resolution plan including:
1. Immediate passenger rebooking strategy
2. Hotel/meal voucher arrangements
3. Alternative routing options
4. Staff deployment (customer care desk, gate agents)
5. Partner airline coordination if needed

Disruption: {info}"""
    )

    return {
        "resolution_plan": response.content,
        "actions_taken": ["Emergency response: full protocol activated"]
    }


def standard_response_node(state: AgentState) -> dict:
    """Handle medium-severity disruptions with standard procedures."""
    info = state["structured_info"]

    response = model.invoke(
        f"""Standard response for medium-severity disruption.
Create a resolution plan including:
1. Rebooking options for affected connecting passengers
2. Lounge access for premium passengers
3. Updated departure information

Disruption: {info}"""
    )

    return {
        "resolution_plan": response.content,
        "actions_taken": ["Standard response: procedure applied"]
    }


def auto_response_node(state: AgentState) -> dict:
    """Handle low-severity disruptions automatically."""
    info = state["structured_info"]

    response = model.invoke(
        f"""Auto-resolve this minor disruption.
Provide a brief 2-3 sentence resolution plan.

Disruption: {info}"""
    )

    return {
        "resolution_plan": response.content,
        "actions_taken": ["Auto-response: minor issue resolved automatically"]
    }


print("Three response handlers defined: emergency, standard, auto")

In [ ]:
def draft_comms_node(state: AgentState) -> dict:
    """Draft both passenger notification and internal report."""
    resolution = state["resolution_plan"]
    flight = state["flight_number"]
    severity = state["severity"]

    # Generate passenger-facing notification
    passenger_response = model.invoke(
        f"""Write a passenger SMS notification for flight {flight}.
Severity: {severity}. Tone: {'urgent and empathetic' if severity == 'high' else 'friendly and clear'}.
Max 3 sentences. Include actionable next steps.

Resolution plan:
{resolution}"""
    )

    # Generate internal operations report
    internal_response = model.invoke(
        f"""Write a brief internal operations report for flight {flight}.
Include: severity, actions taken, current status, next steps.
Format: bullet points, concise, factual.

Resolution plan:
{resolution}"""
    )

    return {
        "passenger_notification": passenger_response.content,
        "internal_report": internal_response.content,
        "actions_taken": ["Communications: passenger notification and internal report drafted"]
    }


print("draft_comms_node: creates both passenger and internal communications")

In [ ]:
def quality_check_node(state: AgentState) -> dict:
    """Review the communications for quality and completeness."""
    notification = state["passenger_notification"]
    report = state["internal_report"]
    severity = state["severity"]

    response = model.invoke(
        f"""Review these communications for a {severity}-severity disruption.

PASSENGER NOTIFICATION:
{notification}

INTERNAL REPORT:
{report}

Rate quality 1-10 and provide brief feedback. Respond in EXACT format:
SCORE: [1-10]
FEEDBACK: [your feedback]"""
    )

    content = response.content
    score = 7
    feedback = content

    for line in content.split("\n"):
        line_upper = line.strip().upper()
        if line_upper.startswith("SCORE:"):
            try:
                score = int(line.split(":", 1)[1].strip().split("/")[0].strip())
            except (ValueError, IndexError):
                score = 7
        elif line_upper.startswith("FEEDBACK:"):
            feedback = line.split(":", 1)[1].strip()

    return {
        "quality_score": score,
        "quality_feedback": feedback,
        "actions_taken": [f"Quality check: score={score}/10"]
    }


print("quality_check_node: reviews and scores the final output")

## 4. Routing Function

In [ ]:
def route_by_severity(state: AgentState) -> str:
    """Route to the appropriate response handler based on severity."""
    severity = state["severity"]
    priority = state["priority_score"]

    print(f"  Routing: severity={severity}, priority={priority}")

    if severity == "high":
        return "emergency_response"
    elif severity == "medium":
        return "standard_response"
    else:
        return "auto_response"


print("Routing function: high → emergency, medium → standard, low → auto")

## 5. Assemble the Graph

In [ ]:
# Build the complete agent graph
builder = StateGraph(AgentState)

# Add all nodes
builder.add_node("intake", intake_node)
builder.add_node("classify", classify_node)
builder.add_node("emergency_response", emergency_response_node)
builder.add_node("standard_response", standard_response_node)
builder.add_node("auto_response", auto_response_node)
builder.add_node("draft_comms", draft_comms_node)
builder.add_node("quality_check", quality_check_node)

# Linear edges: START → intake → classify
builder.add_edge(START, "intake")
builder.add_edge("intake", "classify")

# Conditional edge: classify → (severity-based routing)
builder.add_conditional_edges(
    "classify",
    route_by_severity,
    {
        "emergency_response": "emergency_response",
        "standard_response": "standard_response",
        "auto_response": "auto_response",
    }
)

# All response handlers converge to draft_comms
builder.add_edge("emergency_response", "draft_comms")
builder.add_edge("standard_response", "draft_comms")
builder.add_edge("auto_response", "draft_comms")

# draft_comms → quality_check → END
builder.add_edge("draft_comms", "quality_check")
builder.add_edge("quality_check", END)

# Compile the agent
agent = builder.compile()
print("Agent compiled successfully!")

In [ ]:
# Visualize the complete agent architecture
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print("Agent architecture:")
    print("START → intake → classify → [route]")
    print("  ├── high   → emergency_response ─┐")
    print("  ├── medium → standard_response ──├── draft_comms → quality_check → END")
    print("  └── low    → auto_response ──────┘")

## 6. Run the Complete Agent

In [ ]:
# Full test: a cancellation scenario
test_report = {
    "raw_report": """DISRUPTION REPORT — 2026-04-02 18:45 UTC

Flight TK1920 IST→JFK has been CANCELLED due to a hydraulic system fault
detected during pre-flight checks. Aircraft: Boeing 777-300ER.

Passengers: 340 (42 business, 298 economy)
Connecting passengers: 95 at JFK (various domestic US flights)
Unaccompanied minors: 3
Wheelchair passengers: 7

Next available TK flight IST→JFK: TK1922 (tomorrow, 174 seats available)
Codeshare options: LH via FRA (next day, 45 seats), AF via CDG (next day, 38 seats)

Current airport situation: Terminal 1 hotel desk is open. Meal vouchers available."""
}

print("Running the complete agent...\n")

# Stream to watch each step execute
final_state = None
for step in agent.stream(test_report):
    node_name = list(step.keys())[0]
    print(f">>> {node_name} <<<")
    final_state = step[node_name]

In [ ]:
# Get the complete final state
result = agent.invoke(test_report)

# Print a clean summary
print("=" * 60)
print("AGENT EXECUTION COMPLETE")
print("=" * 60)

print(f"\nFlight: {result['flight_number']}")
print(f"Type: {result['disruption_type']}")
print(f"Severity: {result['severity']} (Priority: {result['priority_score']}/10)")

print(f"\n--- Passenger Notification ---")
print(result["passenger_notification"])

print(f"\n--- Internal Report ---")
print(result["internal_report"])

print(f"\n--- Quality ---")
print(f"Score: {result['quality_score']}/10")
print(f"Feedback: {result['quality_feedback']}")

print(f"\n--- Actions Log ---")
for i, action in enumerate(result["actions_taken"], 1):
    print(f"  {i}. {action}")

## 7. Test with a Minor Disruption

The same agent should handle minor issues with a lighter touch.

In [ ]:
# Test with a minor delay
minor_report = {
    "raw_report": """DISRUPTION REPORT — 2026-04-02 08:15 UTC

Flight TK1234 IST→CDG delayed 25 minutes.
Reason: Late pushback due to congestion on taxiway.
Passengers: 180. No connecting issues expected.
New estimated departure: 09:55."""
}

result_minor = agent.invoke(minor_report)

print(f"Flight: {result_minor['flight_number']}")
print(f"Severity: {result_minor['severity']} (Priority: {result_minor['priority_score']}/10)")
print(f"\nNotification: {result_minor['passenger_notification']}")
print(f"\nActions taken:")
for action in result_minor["actions_taken"]:
    print(f"  - {action}")

---

## YOUR TURN: Extend the Agent

Add a **rebooking_node** that runs after the response handler (but before draft_comms) for high and medium severity disruptions. It should:
1. Analyze the resolution plan
2. Generate specific rebooking recommendations (which flights, how many seats needed)
3. Add to the state as `rebooking_plan`

Low severity disruptions should skip this node entirely.

This requires modifying the graph structure — think about where the conditional edges need to go.

In [ ]:
# YOUR CODE HERE

# Hints:
# 1. Add rebooking_plan: str to the state
# 2. Create rebooking_node that reads resolution_plan and generates rebooking details
# 3. Route: emergency_response → rebooking → draft_comms
#           standard_response → rebooking → draft_comms
#           auto_response → draft_comms (skip rebooking)
# 4. Update draft_comms to include rebooking_plan in the internal report



## Key Takeaways

- **Agents** combine everything: state, nodes, conditional routing, error handling
- **Annotated lists** with `operator.add` let multiple nodes append to the same field
- **Structured parsing** from LLM outputs requires defensive coding (defaults, try/except)
- **Quality gates** (quality_check_node) add a review step — useful for production
- **Streaming** (`graph.stream()`) is essential for debugging multi-node agents

**Congratulations!** You've built a complete AI agent. From here, you can add more nodes, more conditions, looping (retry until quality score > 8), and external API calls.